In [0]:

import copy
import requests
import z_pipelineUtils as z_pU

# File that stores the most recent run details of the pipeline
PIPELINE_STATE = "/Volumes/nc_pipeline/default/pipeline_data/pipeline_state.json"

saved_state = z_pU.readStateJson(PIPELINE_STATE)
live_state = copy.deepcopy(saved_state)

for voter_file in saved_state:
    url = saved_state[voter_file]["url"]
    etag = saved_state[voter_file]["etag"]
    downloaded = saved_state[voter_file]["downloaded"]
    unzipped = saved_state[voter_file]["unzipped"]
    tested = saved_state[voter_file]["tested"]
    archived = saved_state[voter_file]["archived"]
    processed = saved_state[voter_file]["processed"]
    print(f"voter_file: {voter_file}")
    print(f"url: {url}")
    print(f"etag: {etag}")
    print(f"downloaded: {downloaded}")
    print()

    print(f"Accessing {url}")
    r = requests.get(url)
    r_status = r.status_code
    r_head = r.headers
    r_eTag = r_head["ETag"]
    r_contentLength = int(r_head["Content-Length"])
    print(f"r_status: {r_status}")
    print(f"r_head: {r_head}")
    print(f"r_eTag: {r_eTag}")
    print(f"r_contentLength: {r_contentLength}")
    print()

    if r_status == requests.codes.ok:
        if etag != r_eTag:
            live_state[voter_file]["etag"] = r_eTag
            live_state[voter_file]["content_length"] = r_contentLength
            live_state[voter_file]["downloaded"] = False
            live_state[voter_file]["unzipped"] = False
            live_state[voter_file]["tested"] = False
            live_state[voter_file]["archived"] = False
            live_state[voter_file]["processed"] = False
            z_pU.updateStateJson(live_state, PIPELINE_STATE)
            print(f"There is a new file for {voter_file}. The pipeline for {voter_file} will continue.")
        else:
            print(f"There is NOT a new file available for {voter_file}.")
            if (etag == r_eTag) and (processed is False):
                print(f"However, the pipeline did not complete during the prior run.")
                if downloaded is False:
                    print("The pipeline was interrupted prior to downloading the new file.")
                elif unzipped is False:
                    print("The pipeline was interrupted prior to unzipping the download of the new file.")
                elif tested is False:
                    print("The pipeline was interrupted prior to tests on the new file completing.")
                elif archived is False:
                    print("The pipeline was interrupted prior to archiving the existing database tables.")
                else:
                    print("The pipeline was interrupted prior to processing (identifying and appending new data) the new files.")
                print("The pipeline will resume from there.")
    else:
        r.raise_for_status()
